# Meter API — Bundled Query Demo

This notebook demonstrates **GraphQL query bundling**: combining multiple resource types into a single HTTP request to minimise API calls and stay within the 500 requests-per-minute rate limit.

## Why Bundle?

GraphQL's key advantage over REST APIs is that you can request multiple independent resource types in a single round-trip. The Meter API documentation explicitly recommends this approach:

> *"Batch related data into a single query to reduce the number of requests you make and avoid hitting rate limits."*

## How Bundling Works

A GraphQL operation can contain multiple top-level fields. When two fields share the same name (e.g., two `networkClients` calls for different networks) you use an **alias** to distinguish them:

```graphql
{
  primaryClients:   networkClients(networkUUID: "uuid-1") { macAddress }
  secondaryClients: networkClients(networkUUID: "uuid-2") { macAddress }
}
```

This results in **ONE HTTP request** returning **BOTH datasets** under their aliased keys.

## Request Count Comparison

| Demo | REST requests needed | GraphQL requests used |
|---|---|---|
| Network Overview Bundle | 4 (company + clients + uplinks + events) | 1 |
| Metrics Bundle | 3 (quality + throughput + switch ports) | 1 |
| Inventory Bundle | 4 (devices + clients + SSIDs + VLANs) | 1 |
| **Total** | **11** | **3** |

## Setup — Imports and Configuration

In [ ]:
import json
import time
import requests
import config

API_URL             = config.API_URL
API_TOKEN           = config.API_TOKEN
COMPANY_SLUG        = config.COMPANY_SLUG
NETWORK_UUID        = config.NETWORK_UUID
VIRTUAL_DEVICE_UUID = config.VIRTUAL_DEVICE_UUID

HEADERS = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_TOKEN}",
}

print(f"Endpoint : {API_URL}")
print(f"Network  : {NETWORK_UUID}")

## Core Request Helper

`run_query` sends a GraphQL query (which may contain multiple top-level fields) and returns the parsed response along with the response headers.

In [ ]:
def run_query(query: str) -> tuple:
    """
    Execute a GraphQL query and return (parsed_response, response_headers).

    Args:
        query: GraphQL query string (may contain multiple top-level fields).

    Returns:
        Tuple of (response_dict, headers_dict).
    """
    response = requests.post(
        API_URL, headers=HEADERS, json={"query": query}, timeout=60
    )
    response.raise_for_status()
    return response.json(), dict(response.headers)


def summarise(label: str, value) -> None:
    """Print a concise summary line for a single result dataset."""
    if isinstance(value, list):
        print(f"  {label:<30} {len(value):>4} item(s)")
    elif isinstance(value, dict):
        scalars = {k: v for k, v in value.items() if not isinstance(v, (dict, list))}
        print(f"  {label:<30} {scalars}")
    elif value is None:
        print(f"  {label:<30} (null)")
    else:
        print(f"  {label:<30} {value}")

## Bundle 1 — Network Overview (4 resources in 1 request)

Fetches a full network snapshot by bundling four independent resource types:

| Alias | Query | Description |
|---|---|---|
| `companyInfo` | `companyBySlug` | Basic company details |
| *(none)* | `networkClients` | All active clients |
| `uplinkInterfaces` | `uplinkPhyInterfacesForNetwork` | WAN uplink port status |
| `eventLog` | `recentEventLogEventsPage` | Most recent 10 events |

Without bundling this would require **4 separate HTTP requests**.

In [ ]:
def network_overview_bundle(company_slug: str, network_uuid: str) -> dict:
    """
    Fetch a full network snapshot in a single HTTP request.

    Bundles: companyInfo, networkClients, uplinkInterfaces, eventLog.
    Without bundling: 4 requests. With bundling: 1 request.
    """
    query = f"""
    {{
      companyInfo: companyBySlug(slug: "{company_slug}") {{
        uuid
        name
        slug
        isCustomer
        websiteDomain
      }}

      networkClients(networkUUID: "{network_uuid}") {{
        macAddress
        ip
        clientName
        isWireless
        signal
        lastSeen
        connectedVLAN {{
          name
          vlanID
        }}
        connectedSSID {{
          ssid
        }}
      }}

      uplinkInterfaces: uplinkPhyInterfacesForNetwork(networkUUID: "{network_uuid}") {{
        UUID
        label
        portNumber
        isEnabled
        isUplinkActive
        portSpeedMbps
        nativeVLAN {{
          name
          vlanID
        }}
      }}

      eventLog: recentEventLogEventsPage(networkUUID: "{network_uuid}", limit: 10) {{
        total
        events {{
          eventType
          eventTypeAPIName
          generatedAt
          networkUUID
        }}
      }}
    }}
    """
    result, headers = run_query(query)
    return result

In [ ]:
print("[1/3] Network Overview Bundle")
print("      companyInfo  +  networkClients  +  uplinkInterfaces  +  eventLog")

t0 = time.monotonic()
result1 = network_overview_bundle(COMPANY_SLUG, NETWORK_UUID)
elapsed1 = time.monotonic() - t0

print(f"\n→ 1 HTTP request completed in {elapsed1 * 1000:.0f} ms")

if "errors" in result1:
    print("GraphQL errors:")
    for err in result1["errors"]:
        print(f"  ✗ [{err.get('extensions', {}).get('code', '?')}] {err.get('message')}")
else:
    data1 = result1.get("data", {})
    print("\nResults:")
    for key, value in data1.items():
        summarise(key, value)

    company = data1.get("companyInfo", {})
    events  = data1.get("eventLog", {})
    if company:
        print(f"\n  Company   : {company.get('name')} ({company.get('slug')})")
    if events:
        print(f"  Events    : {events.get('total')} total, last {len(events.get('events', []))} shown")

## Bundle 2 — Metrics (3 metrics in 1 request)

Fetches three time-series metrics datasets with a shared time filter (last 4 hours, 5-minute resolution):

| Alias | Query | Description |
|---|---|---|
| `uplinkQuality` | `networksUplinkQualities` | WAN link quality scores per interface |
| `uplinkThroughput` | `networkUplinkThroughput` | Upload/download bandwidth per interface |
| *(none)* | `switchPortStats` | Cumulative per-port traffic counters |

The `MetricsFilterInput`:
- `durationSeconds: 14400` — look back 4 hours
- `stepSeconds: 300` — aggregate into 5-minute buckets

In [ ]:
def metrics_bundle(network_uuid: str, virtual_device_uuid: str) -> dict:
    """
    Fetch three time-series metrics datasets in a single HTTP request.

    Bundles: uplinkQuality, uplinkThroughput, switchPortStats.
    Without bundling: 3 requests. With bundling: 1 request.
    """
    query = f"""
    {{
      uplinkQuality: networksUplinkQualities(
        networkUUIDs: ["{network_uuid}"],
        filter: {{ durationSeconds: 14400, stepSeconds: 300 }}
      ) {{
        metadata {{
          minValue
          maxValue
        }}
        values {{
          timestamp
          value
          phyInterfaceUUID
          networkUUID
        }}
      }}

      uplinkThroughput: networkUplinkThroughput(
        networkUUID: "{network_uuid}",
        filter: {{ durationSeconds: 14400, stepSeconds: 300 }}
      ) {{
        metadata {{
          minValue
          maxValue
        }}
        values {{
          timestamp
          value
          direction
          phyInterfaceUUID
        }}
      }}

      switchPortStats(virtualDeviceUUID: "{virtual_device_uuid}") {{
        portNumber
        totalRxBytes
        totalTxBytes
        totalRxPackets
        totalTxPackets
        errorRxPackets
        errorTxPackets
      }}
    }}
    """
    result, _ = run_query(query)
    return result

In [ ]:
print("[2/3] Metrics Bundle")
print("      uplinkQuality  +  uplinkThroughput  +  switchPortStats")

t0 = time.monotonic()
result2 = metrics_bundle(NETWORK_UUID, VIRTUAL_DEVICE_UUID)
elapsed2 = time.monotonic() - t0

print(f"\n→ 1 HTTP request completed in {elapsed2 * 1000:.0f} ms")

if "errors" in result2:
    for err in result2["errors"]:
        print(f"  ✗ [{err.get('extensions', {}).get('code', '?')}] {err.get('message')}")
else:
    data2 = result2.get("data", {})
    print("\nResults:")
    for key, value in data2.items():
        summarise(key, value)

    quality_data = data2.get("uplinkQuality", [])
    if quality_data:
        all_values = sum(len(q.get("values", [])) for q in quality_data)
        print(f"\n  Uplink quality data points : {all_values}")

## Bundle 3 — Network Inventory (4 lists in 1 request)

Fetches the complete network inventory:

| Alias | Query | Description |
|---|---|---|
| `devices` | `virtualDevicesForNetwork` | All virtual devices |
| `clients` | `networkClients` | All active clients |
| `ssids` | `ssidsForNetwork` | All configured SSIDs |
| `vlans` | `vlans` | All configured VLANs |

Without bundling: **4 requests**. With bundling: **1 request**.

In [ ]:
def inventory_bundle(network_uuid: str) -> dict:
    """
    Fetch the complete network inventory in a single HTTP request.

    Bundles: devices, clients, ssids, vlans.
    Without bundling: 4 requests. With bundling: 1 request.
    """
    query = f"""
    {{
      devices: virtualDevicesForNetwork(networkUUID: "{network_uuid}") {{
        UUID
        label
        deviceType
        deviceModel
        isOnline
      }}

      clients: networkClients(networkUUID: "{network_uuid}") {{
        macAddress
        ip
        clientName
        isWireless
        lastSeen
      }}

      ssids: ssidsForNetwork(networkUUID: "{network_uuid}") {{
        UUID
        ssid
        isEnabled
        isGuest
        isHidden
        encryptionProtocol
      }}

      vlans: vlans(networkUUID: "{network_uuid}") {{
        UUID
        name
        vlanID
        isEnabled
        isDefault
      }}
    }}
    """
    result, _ = run_query(query)
    return result

In [ ]:
print("[3/3] Inventory Bundle")
print("      devices  +  clients  +  ssids  +  vlans")

t0 = time.monotonic()
result3 = inventory_bundle(NETWORK_UUID)
elapsed3 = time.monotonic() - t0

print(f"\n→ 1 HTTP request completed in {elapsed3 * 1000:.0f} ms")

if "errors" in result3:
    for err in result3["errors"]:
        print(f"  ✗ [{err.get('extensions', {}).get('code', '?')}] {err.get('message')}")
else:
    data3 = result3.get("data", {})
    print("\nResults:")
    for key, value in data3.items():
        summarise(key, value)

    ssids = data3.get("ssids", [])
    if ssids:
        print(f"\n  SSIDs:")
        for ssid in ssids:
            status = "enabled" if ssid.get("isEnabled") else "disabled"
            guest  = " [guest]" if ssid.get("isGuest") else ""
            print(f"    • {ssid.get('ssid') or '':<30} {ssid.get('encryptionProtocol') or '':<20} {status}{guest}")

## Bundle 4 — Multi-Network Clients (same field aliased twice)

Demonstrates aliasing the **same query field** twice with different arguments. Both `networkClients` calls share the same field name, so aliases are required to distinguish them in the response.

```graphql
{
  primaryNetworkClients:   networkClients(networkUUID: "uuid-1") { ... }
  secondaryNetworkClients: networkClients(networkUUID: "uuid-2") { ... }
  companyClients: networksClients(companyUUID: "...", networkUUIDs: [...]) { ... }
}
```

Without bundling: **3 requests**. With bundling: **1 request**.

In [ ]:
def multi_network_clients_bundle(
    network_uuid_a: str,
    network_uuid_b: str,
    company_uuid: str,
) -> dict:
    """
    Fetch clients from two networks + company-wide totals in one request.

    Demonstrates aliasing the same query field twice with different arguments.
    """
    # Deduplicate so the API receives each UUID at most once
    unique_uuids = list(dict.fromkeys([network_uuid_a, network_uuid_b]))
    uuids_gql = '", "'.join(unique_uuids)

    query = f"""
    {{
      primaryNetworkClients: networkClients(networkUUID: "{network_uuid_a}") {{
        macAddress
        ip
        clientName
        isWireless
      }}

      secondaryNetworkClients: networkClients(networkUUID: "{network_uuid_b}") {{
        macAddress
        ip
        clientName
        isWireless
      }}

      companyClients: networksClients(
        companyUUID: "{company_uuid}",
        networkUUIDs: ["{uuids_gql}"]
      ) {{
        macAddress
        ip
        clientName
        isWireless
        lastSeen
      }}
    }}
    """
    result, _ = run_query(query)
    return result

In [ ]:
# Uses the same NETWORK_UUID for both "networks" to demonstrate alias syntax
# In production, pass two different network UUIDs
COMPANY_UUID = config.COMPANY_UUID
NETWORK_UUID = config.NETWORK_UUID

t0 = time.monotonic()
result4 = multi_network_clients_bundle(NETWORK_UUID, NETWORK_UUID, COMPANY_UUID)
elapsed4 = time.monotonic() - t0

print(f"Multi-network bundle completed in {elapsed4 * 1000:.0f} ms")

if "errors" in result4:
    for err in result4["errors"]:
        print(f"  ✗ [{err.get('extensions', {}).get('code', '?')}] {err.get('message')}")
else:
    data4 = result4.get("data", {})
    print("\nResults (same network aliased twice + company-wide):")
    for key, value in data4.items():
        summarise(key, value)

## Summary

In [ ]:
total_elapsed = elapsed1 + elapsed2 + elapsed3 + elapsed4

print("=" * 65)
print("  Summary")
print("=" * 65)
print(f"  Total HTTP requests  : 4")
print(f"  Equivalent REST calls: 14+")
print(f"  Total elapsed time   : {total_elapsed * 1000:.0f} ms")
print()
print("  Each bundle used GraphQL aliases to pack multiple queries into")
print("  one request — no extra round-trips, no extra rate-limit cost.")
print()
print("  Key rule: when the same field name appears twice in a query,")
print("  use aliases to give each occurrence a unique response key.")